In [0]:
df = spark.sql("select p.* from bronzelayer.policydata p inner join bronzelayer.customer c on p.customer_id =c.customer_id where p.customer_id is not null and p.policy_id is not null and p.merge_flag = False and p.premium > 0 and p.coverage_amount > 0 and p.end_date > p.start_date")

display(df)

policy_id,policy_type,customer_id,start_date,end_date,premium,coverage_amount,merge_flag
3,Auto,1609,2023-02-07T09:18:35Z,2024-02-28T05:09:11Z,38737.0,304308.0,false
4,Health,1021,2022-09-05T02:52:10Z,2024-02-28T17:26:41Z,117350.0,651873.0,false
5,Health,1893,2022-11-27T08:26:02Z,2023-08-18T15:21:43Z,156670.0,899773.0,false
6,Health,1536,2022-11-07T06:29:12Z,2023-08-02T16:24:35Z,38190.0,380038.0,false
8,Auto,1612,2023-02-03T22:11:45Z,2023-12-25T04:14:56Z,165981.0,714562.0,false
9,Health,1350,2023-01-25T13:10:50Z,2023-10-19T09:32:35Z,195090.0,273053.0,false
10,Auto,1486,2023-02-27T09:52:02Z,2023-06-14T03:48:11Z,95109.0,1078838.0,false
12,Auto,1768,2022-06-04T08:41:44Z,2023-11-08T20:15:51Z,91474.0,1257396.0,false
13,Health,1174,2022-07-06T16:26:40Z,2023-08-29T02:11:07Z,142738.0,961892.0,false
16,Auto,1406,2022-05-03T06:22:22Z,2024-01-29T21:48:44Z,78603.0,711414.0,false


In [0]:
df.createOrReplaceTempView("clean_policy")

In [0]:
%sql
MERGE INTO silverlayer.policydata T USING clean_policy S ON T.policy_id = s.policy_id WHEN MATCHED THEN UPDATE SET T.policy_type = s.policy_type , T.start_date = s.start_date, T.end_date = s.end_date, T.premium = s.premium, T.coverage_amount = s.coverage_amount, T.merge_timestamp = current_timestamp()
WHEN NOT MATCHED THEN INSERT (policy_id, policy_type, start_date, end_date, premium, coverage_amount,merge_timestamp)
values(s.policy_id,s.policy_type, s.start_date, s.end_date, s.premium,s.coverage_amount, current_timestamp())


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
765,765,0,0


In [0]:
%sql
update bronzelayer.policydata set merge_flag = true where merge_flag = false

num_affected_rows
1000
